# IT135IU Group 3, Topic 8: D3 Analysis
**Decision question:** which film genres should the Content Marketing and Promotion Team promote to each gender x age-group segment, instead of promoting the same genres to everyone? **Data:** MovieLens 1M (GroupLens).

| No. | Role | Name | Student ID |
|-----|------|------|------------|
| 1 | Product owner | Truc Phuong | ITDSIU25031 |
| 2 | Data engineer | Le Quan | ITDSIU25033 |
| 3 | Data analyst A | Que | ITDSIU25036 |
| 4 | Data analyst B | Hoang Quan | ITDSIU25035 |
| 5 | Visualisation lead | Tuyen | ITDSIU23028 |
| 6 | Reproducibility lead | Trung | ITDSIU24054 |
| 7 | Ethics and privacy officer | Cuong | ITDSIU25008 |

## Source and terms of use
- **Dataset:** MovieLens 1M, [grouplens.org/datasets/movielens/1m/](https://grouplens.org/datasets/movielens/1m/) (file loaded in Step 1 from [files.grouplens.org/datasets/movielens/ml-1m.zip](https://files.grouplens.org/datasets/movielens/ml-1m.zip))
- **Publisher:** GroupLens Research, University of Minnesota
- **Date of retrieval:** 28/09/2026
- **Terms of use** (copied from the README in ml-1m.zip):
  Neither the University of Minnesota nor any of the researchers involved can guarantee the correctness of the data, its suitability for any particular purpose, or the validity of results based on the use of the data set. The data set may be used for any research purposes under the following conditions:
  - The user may not state or imply any endorsement from the University of Minnesota or the GroupLens Research Group.
  - The user must acknowledge the use of the data set in publications resulting from the use of the data set (see below for citation information).
  - The user may not redistribute the data without separate permission.
  - The user may not use this information for any commercial or revenue-bearing purposes without first obtaining permission from a faculty member of the GroupLens Research Project at the University of Minnesota.

**Citation:**
> F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. *ACM Transactions on Interactive Intelligent Systems (TiiS)* 5, 4, Article 19 (December 2015), 19 pages. DOI: [http://dx.doi.org/10.1145/2827872](http://dx.doi.org/10.1145/2827872)

## Terms used in this notebook
- **segment**: one gender × age group, e.g. F 25-34 (2 × 7 = 14 segments).
- **cell**: one segment × genre pair, e.g. F 25-34 × Drama (14 × 18 = 252 cells). Code blocks are always referred to by their Step number, never as cells.
- **tier**: reliability label of a cell, based on its number of distinct users.
- **satisfaction rate**: share of a cell's ratings that are 4 or 5 stars.

## Terms used in this notebook
- **Segment:** one gender × age group (2 × 7 = 14 segments).
- **Cell:** one segment × genre pair (14 × 18 = 252 cells).
- **Step:** one code block of this notebook. Steps 1 to 9 are carried over unchanged from D2. D3 Steps are named after the analysis steps: Step B1 to Step B9 (analysis), Step B10.1 to Step B10.4 (charts), Step Q1 and Step Q2 (descriptive and summary tables).
- **Satisfaction rate:** share of a cell's ratings that are 4 or 5 stars (rating-level).
- **Tier:** a cell is *reliable* with at least 100 distinct users, *indicative* otherwise. Tiers are computed once on the full data (Step 7) and kept on both halves.
- **Discovery half / validation half:** users of each segment are split at random into two halves (Step B1). Genres are chosen on the discovery half and tested on the validation half.

### Step 1: Load the Three Raw Tables
Load the raw files (`ratings.dat`, `users.dat`, and `movies.dat`) from the MovieLens zip file. The file is downloaded from GroupLens; if the download fails three times, a local copy in `data/ml-1m.zip` is used (cleaning log C19).

In [1]:
import io
import ssl
import time
import urllib.request
import zipfile
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import re

SEED = 42
np.random.seed(SEED)

URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
LOCAL_ZIP = Path("data/ml-1m.zip")  # used only if the download fails; never committed
READ: dict[str, Any] = {
    "sep": "::",
    "engine": "python",
    "encoding": "latin-1",
    "header": None,
}

AGE_LABEL = {
    1: "Under 18",
    18: "18-24",
    25: "25-34",
    35: "35-44",
    45: "45-49",
    50: "50-55",
    56: "56+",
}

def load(archive: zipfile.ZipFile, name: str, cols: list[str]) -> pd.DataFrame:
    """Read one "::" separated member straight out of the archive."""
    data = io.BytesIO(archive.read(f"ml-1m/{name}"))
    return pd.read_csv(data, names=cols, **READ)


def fetch_zip(url: str, local: Path, tries: int = 3, wait: int = 5) -> zipfile.ZipFile:
    """Download the archive with certifi certificates; fall back to a local copy.

    Certificate verification is never turned off. If every attempt fails and
    there is no local copy, stop with instructions for a manual download.
    """
    try:
        import certifi
        context = ssl.create_default_context(cafile=certifi.where())
    except ImportError:
        context = ssl.create_default_context()

    last_error = None
    for attempt in range(1, tries + 1):
        try:
            with urllib.request.urlopen(url, context=context, timeout=60) as resp:
                data = resp.read()
            print(f"source : {url} (download, attempt {attempt})")
            return zipfile.ZipFile(io.BytesIO(data))
        except Exception as err:  # network or certificate error
            last_error = err
            print(f"download attempt {attempt} failed: {type(err).__name__}: {err}")
            if attempt < tries:
                time.sleep(wait)

    if local.exists():
        print(f"source : local copy {local}")
        return zipfile.ZipFile(local)

    raise RuntimeError(
        f"Could not download {url} ({last_error}) and found no local copy at {local}. "
        "Download ml-1m.zip by hand from https://grouplens.org/datasets/movielens/1m/ "
        "and put it in a folder named data/ next to this notebook."
    )

print("STEP 1  load the three raw tables")
zf = fetch_zip(URL, LOCAL_ZIP)
print("members inside zip:")
for member in zf.namelist():
    print(f"  - {member}")

ratings = load(zf, "ratings.dat", ["user_id", "movie_id", "rating", "timestamp"])
users = load(zf, "users.dat", ["user_id", "gender", "age", "occupation", "zip"])
movies = load(zf, "movies.dat", ["movie_id", "title", "genres"])

STEP 1  load the three raw tables


download attempt 1 failed: URLError: <urlopen error Tunnel connection failed: 403 Forbidden>


download attempt 2 failed: URLError: <urlopen error Tunnel connection failed: 403 Forbidden>


download attempt 3 failed: URLError: <urlopen error Tunnel connection failed: 403 Forbidden>
source : local copy data/ml-1m.zip
members inside zip:
  - ml-1m/
  - ml-1m/movies.dat
  - ml-1m/ratings.dat
  - ml-1m/README
  - ml-1m/users.dat


### Step 2: Missing Values, Keys, and Orphan Rows
Validate the integrity of our tables by checking for missing values, confirming primary keys are unique, and identifying any orphaned foreign keys.

In [2]:
print("STEP 2  missing values, keys and orphan rows")

for name, table in [
    ("ratings", ratings),
    ("users", users),
    ("movies", movies),
]:
    print(f"\n{name}: shape={table.shape}")
    print(table.isna().sum().to_string())

print("\nprimary key uniqueness")
print(f"  users.user_id   : {users['user_id'].is_unique}")
print(f"  movies.movie_id : {movies['movie_id'].is_unique}")

bad_user = ~ratings["user_id"].isin(users["user_id"])
bad_movie = ~ratings["movie_id"].isin(movies["movie_id"])

n_bad_user = bad_user.sum()
n_bad_movie = bad_movie.sum()
d_bad_user = ratings.loc[bad_user, "user_id"].nunique()
d_bad_movie = ratings.loc[bad_movie, "movie_id"].nunique()

print("\norphan foreign keys in ratings")
print(f"  rows, unknown user_id     : {n_bad_user:,}")
print(f"  rows, unknown movie_id    : {n_bad_movie:,}")
print(f"  distinct unknown user_id  : {d_bad_user}")
print(f"  distinct unknown movie_id : {d_bad_movie}")

STEP 2  missing values, keys and orphan rows

ratings: shape=(1000209, 4)
user_id      0
movie_id     0
rating       0
timestamp    0

users: shape=(6040, 5)
user_id       0
gender        0
age           0
occupation    0
zip           0

movies: shape=(3883, 3)
movie_id    0
title       0
genres      0

primary key uniqueness
  users.user_id   : True
  movies.movie_id : True

orphan foreign keys in ratings
  rows, unknown user_id     : 0
  rows, unknown movie_id    : 0
  distinct unknown user_id  : 0
  distinct unknown movie_id : 0


### Step 2b: Data Profile of the Three Raw Tables
Rows, columns, types, missing values, distinct values and ranges for each raw table (the profile table required by the brief).

In [3]:
print("STEP 2b  data profile of the raw tables")

def profile(table: pd.DataFrame, name: str) -> pd.DataFrame:
    """One row per column: dtype, missing, distinct values and numeric range."""
    rows = []
    for col in table.columns:
        s = table[col]
        numeric = pd.api.types.is_numeric_dtype(s)
        rows.append(
            {
                "column": col,
                "dtype": str(s.dtype),
                "missing": int(s.isna().sum()),
                "unique": int(s.nunique()),
                "min": s.min() if numeric else "-",
                "max": s.max() if numeric else "-",
            }
        )
    print(f"\n{name}: {table.shape[0]:,} rows x {table.shape[1]} columns")
    out = pd.DataFrame(rows)
    print(out.to_string(index=False))
    return out

for name, table in [("ratings", ratings), ("users", users), ("movies", movies)]:
    profile(table, name)

STEP 2b  data profile of the raw tables

ratings: 1,000,209 rows x 4 columns
   column dtype  missing  unique       min        max
  user_id int64        0    6040         1       6040
 movie_id int64        0    3706         1       3952
   rating int64        0       5         1          5
timestamp int64        0  458455 956703932 1046454590

users: 6,040 rows x 5 columns
    column dtype  missing  unique min  max
   user_id int64        0    6040   1 6040
    gender   str        0       2   -    -
       age int64        0       7   1   56
occupation int64        0      21   0   20
       zip   str        0    3439   -    -

movies: 3,883 rows x 3 columns
  column dtype  missing  unique min  max
movie_id int64        0    3883   1 3952
   title   str        0    3883   -    -
  genres   str        0     301   -    -


### Step 2c: Duplicate, Validity and Coverage Checks
Duplicate rating pairs, valid value ranges, rating period, and films that no one rated.

In [4]:
print("STEP 2c  duplicate, validity and coverage checks")

n_dup_pairs = ratings.duplicated(subset=["user_id", "movie_id"]).sum()
n_dup_rows = ratings.duplicated().sum()
print("duplicate rows in ratings")
print(f"  repeated (user_id, movie_id) pairs : {n_dup_pairs:,}")
print(f"  fully identical rows               : {n_dup_rows:,}")

print("\nvalid value ranges")
print(f"  rating values : {[int(x) for x in sorted(ratings['rating'].unique())]}")
print(f"  gender codes  : {sorted(users['gender'].unique())}")
stamps = pd.to_datetime(ratings["timestamp"], unit="s", utc=True)
print(f"  first rating  : {stamps.min()}")
print(f"  last rating   : {stamps.max()}")

rated_ids = set(ratings["movie_id"])
n_unrated = (~movies["movie_id"].isin(rated_ids)).sum()
print("\nfilm coverage")
print(f"  films in movies.dat      : {len(movies):,}")
print(f"  films with >= 1 rating   : {len(rated_ids):,}")
print(f"  films never rated        : {n_unrated:,}")
print(f"  highest movie_id         : {movies['movie_id'].max():,}")
print(f"  unused ids up to highest : {movies['movie_id'].max() - len(movies):,}")
print(f"  fewest ratings by one user : {ratings.groupby('user_id').size().min()}")

STEP 2c  duplicate, validity and coverage checks


duplicate rows in ratings
  repeated (user_id, movie_id) pairs : 0
  fully identical rows               : 0

valid value ranges
  rating values : [1, 2, 3, 4, 5]
  gender codes  : ['F', 'M']


  first rating  : 2000-04-25 23:05:32+00:00
  last rating   : 2003-02-28 17:49:50+00:00

film coverage
  films in movies.dat      : 3,883
  films with >= 1 rating   : 3,706
  films never rated        : 177
  highest movie_id         : 3,952
  unused ids up to highest : 69
  fewest ratings by one user : 20


### Step 2d: Additional Checks (Data Analyst A)
Three checks taken from the independent notebook of Data Analyst A (Que): row counts against the README, ratings dated before the film's release year, and ratings entered in the same second by the same user. Nothing is removed here; the results are recorded in the cleaning log (C17). Two notebooks built separately give the same row counts and checks.

In [5]:
print("STEP 2d (1)  row counts against the README")

readme = re.sub(r"\s+", " ", zf.read("ml-1m/README").decode("latin-1"))
ref_ratings = int(re.search(r"([\d,]+) anonymous ratings", readme).group(1).replace(",", ""))
ref_users = int(re.search(r"made by ([\d,]+) MovieLens users", readme).group(1).replace(",", ""))

print(f"  ratings : README {ref_ratings:,}, file {len(ratings):,}, match {ref_ratings == len(ratings)}")
print(f"  users   : README {ref_users:,}, file {len(users):,}, match {ref_users == len(users)}")

STEP 2d (1)  row counts against the README
  ratings : README 1,000,209, file 1,000,209, match True
  users   : README 6,040, file 6,040, match True


In [6]:
print("STEP 2d (2)  ratings dated before the film's release year")

release_year = (
    movies.set_index("movie_id")["title"]
    .str.extract(r"\((\d{4})\)\s*$", expand=False)
    .astype(int)
)
rated_year = pd.to_datetime(ratings["timestamp"], unit="s", utc=True).dt.year
before = rated_year < ratings["movie_id"].map(release_year)

print(f"  ratings checked           : {len(ratings):,}")
print(f"  rated before release year : {before.sum():,}")

STEP 2d (2)  ratings dated before the film's release year


  ratings checked           : 1,000,209
  rated before release year : 0


In [7]:
print("STEP 2d (3)  ratings entered in the same second by the same user")

per_second = ratings.groupby(["user_id", "timestamp"]).size()
in_burst = int(per_second[per_second > 1].sum())

print(f"  ratings checked                         : {len(ratings):,}")
print(f"  ratings sharing a second with another   : {in_burst:,} ({in_burst / len(ratings):.1%})")
print(f"  largest burst, one user in one second   : {per_second.max()} ratings")

STEP 2d (3)  ratings entered in the same second by the same user


  ratings checked                         : 1,000,209
  ratings sharing a second with another   : 774,028 (77.4%)
  largest burst, one user in one second   : 30 ratings


### Step 3: Merge Into One Working Table
Merge the `ratings`, `users`, and `movies` DataFrames using left joins.

In [8]:
print("STEP 3  merge into one working table")

df = ratings.merge(users, on="user_id", how="left", validate="m:1")
df = df.merge(movies, on="movie_id", how="left", validate="m:1")

assert len(df) == len(ratings), "left merge changed the row count"

print(f"shape     : {df.shape}")
print(f"columns   : {df.columns.tolist()}")
print(f"total NaN : {df.isna().sum().sum()}")

STEP 3  merge into one working table


shape     : (1000209, 10)
columns   : ['user_id', 'movie_id', 'rating', 'timestamp', 'gender', 'age', 'occupation', 'zip', 'title', 'genres']


total NaN : 0


### Step 3b: Drop Personal Columns
`zip` and `occupation` are dropped from the working table: zip code combined with gender and age band could narrow down an individual, and neither column is used in the analysis (as committed in D1).

In [9]:
print("STEP 3b  drop personal columns")

before = df.shape[1]
df = df.drop(columns=["zip", "occupation"])
print(f"columns before : {before}")
print(f"columns after  : {df.shape[1]}")
print(f"columns        : {df.columns.tolist()}")
assert len(df) == len(ratings), "dropping columns changed the row count"

STEP 3b  drop personal columns
columns before : 10
columns after  : 8
columns        : ['user_id', 'movie_id', 'rating', 'timestamp', 'gender', 'age', 'title', 'genres']


### Step 4: Derived Columns (Year and Age Group)
Extract the release year from movie titles and map numeric age codes to readable categories.

In [10]:
print("STEP 4  derived columns, year and age_group")

df["year"] = (
    df["title"].str.extract(r"\((\d{4})\)\s*$", expand=False).astype("Int64")
)

print(f"rows with NaN year : {df['year'].isna().sum()}")
print(f"year.min()         : {df['year'].min()}")
print(f"year.max()         : {df['year'].max()}")

codes = df["age"]
df["age_group"] = codes.map(AGE_LABEL)

assert df["age_group"].isna().sum() == 0, "unmapped age code found"

print(f"\nrows with NaN age_group : {df['age_group'].isna().sum()}")

age_order = [AGE_LABEL[code] for code in sorted(AGE_LABEL)]
print(df["age_group"].value_counts().reindex(age_order).to_string())

STEP 4  derived columns, year and age_group


rows with NaN year : 0
year.min()         : 1919
year.max()         : 2000

rows with NaN age_group : 0
age_group
Under 18     27211
18-24       183536
25-34       395556
35-44       199003
45-49        83633
50-55        72490
56+          38780


In [11]:
print("STEP 4b  release-year coverage")

years_rated = set(df["year"].dropna().astype(int))
years_all = set(
    movies["title"].str.extract(r"\((\d{4})\)\s*$", expand=False).dropna().astype(int)
)
span = range(min(years_all), max(years_all) + 1)
print(f"distinct years, rated films in df : {len(years_rated)}")
print(f"distinct years, all of movies.dat : {len(years_all)}")
print(f"years in the span with no film    : {[y for y in span if y not in years_all]}")

STEP 4b  release-year coverage


distinct years, rated films in df : 81
distinct years, all of movies.dat : 81
years in the span with no film    : [1924]


### Step 5: Genre Breakdown
Explode the concatenated genres so each row represents a single rating-genre pair.

In [12]:
print("STEP 5  genre breakdown")

genres = df.copy()
genres["genre_list"] = genres["genres"].str.split("|")
genres["n_genres"] = genres["genre_list"].str.len()

print("n_genres per rating row")
print(genres["n_genres"].describe().to_string())

exploded = genres.explode("genre_list").rename(columns={"genre_list": "genre"})

print(f"\nrows after explode : {len(exploded):,}")
print(f"ratio to ratings   : {len(exploded) / len(ratings):.4f}")
print(f"distinct genres    : {exploded['genre'].nunique()}")
print(exploded["genre"].value_counts().to_string())

STEP 5  genre breakdown


n_genres per rating row
count    1.000209e+06
mean     2.101376e+00
std      9.768456e-01
min      1.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      3.000000e+00
max      6.000000e+00



rows after explode : 2,101,815
ratio to ratings   : 2.1014
distinct genres    : 18


genre
Comedy         356580
Drama          354529
Action         257457
Thriller       189680
Sci-Fi         157294
Romance        147523
Adventure      133953
Crime           79541
Horror          76386
Children's      72186
War             68527
Animation       43293
Musical         41533
Mystery         40178
Fantasy         36301
Western         20683
Film-Noir       18261
Documentary      7910


In [13]:
print("STEP 5b  labels per film and genre combinations")

per_film_all = movies["genres"].str.count(r"\|") + 1
per_film_rated = per_film_all[movies["movie_id"].isin(rated_ids)]
print(f"distinct genre combinations         : {movies['genres'].nunique()}")
print(f"mean labels per rating row (df)     : {genres['n_genres'].mean():.4f}")
print(f"mean labels per film, all movies    : {per_film_all.mean():.4f}")
print(f"mean labels per film, rated movies  : {per_film_rated.mean():.4f}")

STEP 5b  labels per film and genre combinations
distinct genre combinations         : 301
mean labels per rating row (df)     : 2.1014
mean labels per film, all movies    : 1.6503
mean labels per film, rated movies  : 1.6708


### Step 6: Cell Sparsity
Count distinct users and ratings in each of the 252 cells (segment × genre) to see which cells have thin data support.

In [14]:
print("STEP 6  cell sparsity")

cells = (
    exploded.groupby(["gender", "age_group", "genre"], observed=True)
    .agg(n_ratings=("rating", "size"), n_users=("user_id", "nunique"))
    .reset_index()
    .sort_values("n_users")
)

cells["age_group"] = pd.Categorical(
    cells["age_group"], age_order, ordered=True
)

print(f"cells present    : {len(cells)}")
print(f"full grid 2x7x18 : {2 * 7 * 18}")
print("\nn_users per cell")
print(cells["n_users"].describe().round(1).to_string())

for limit in (30, 50, 100):
    thin = cells[cells["n_users"] < limit]
    share = len(thin) / len(cells)
    print(
        f"\nunder {limit} users: {len(thin)} of {len(cells)} cells ({share:.1%})"
    )
    rows = zip(
        thin["gender"],
        thin["age_group"],
        thin["genre"],
        thin["n_users"],
        thin["n_ratings"],
    )
    for gender, age, genre, n_users, n_ratings in rows:
        label = f"{gender} {age}"
        print(
            f"  {label:<12} {genre:<12} {n_users:>3} users, {n_ratings:>5} ratings"
        )

STEP 6  cell sparsity


cells present    : 252
full grid 2x7x18 : 252

n_users per cell
count     252.0
mean      372.6
std       354.7
min        11.0
25%       133.5
50%       270.5
75%       460.5
max      1538.0

under 30 users: 3 of 252 cells (1.2%)
  F Under 18   Documentary   11 users,    25 ratings
  F Under 18   Film-Noir     26 users,    80 ratings
  F 56+        Documentary   29 users,    72 ratings

under 50 users: 5 of 252 cells (2.0%)
  F Under 18   Documentary   11 users,    25 ratings
  F Under 18   Film-Noir     26 users,    80 ratings
  F 56+        Documentary   29 users,    72 ratings
  F Under 18   Western       35 users,    91 ratings
  M Under 18   Documentary   45 users,   105 ratings

under 100 users: 41 of 252 cells (16.3%)
  F Under 18   Documentary   11 users,    25 ratings
  F Under 18   Film-Noir     26 users,    80 ratings
  F 56+        Documentary   29 users,    72 ratings
  F Under 18   Western       35 users,    91 ratings
  M Under 18   Documentary   45 users,   105 ratings

### Step 7: Freeze the Reliability Threshold into a Tier Column
A cell is **reliable** if it holds at least 100 distinct users, and **indicative** otherwise. The threshold was fixed in D1 (Section 5, Feasibility note) before any satisfaction rate was computed. Cell size is counted in distinct users, not ratings, because one person's ratings are not independent of each other. Indicative cells stay in the cell table and are reported as indicative only; they are never used to rank a segment's genres. D1 allowed such cells to be "pooled or reported as indicative only"; this notebook takes the second option and does not pool.

In [15]:
print("STEP 7  freeze the reliability threshold into a tier column")

MIN_USERS = 100  # fixed in D1, Section 5, before any satisfaction rate was computed

cells["tier"] = np.where(cells["n_users"] >= MIN_USERS, "reliable", "indicative")

tier_counts = cells["tier"].value_counts().reindex(["reliable", "indicative"])
assert tier_counts.sum() == len(cells) == 2 * 7 * 18, "tiers miss some cells"

print(f"threshold : {MIN_USERS} distinct users per cell")
print(tier_counts.to_string())
print(f"reliable share : {tier_counts['reliable'] / len(cells):.1%}")

print("\nreliable cells per segment (out of 18 genres)")
reliable_per_segment = (
    cells.groupby(["gender", "age_group"], observed=False)["tier"]
    .apply(lambda s: int((s == "reliable").sum()))
    .unstack("age_group")
)
print(reliable_per_segment.to_string())

STEP 7  freeze the reliability threshold into a tier column
threshold : 100 distinct users per cell
tier
reliable      211
indicative     41
reliable share : 83.7%

reliable cells per segment (out of 18 genres)
age_group  Under 18  18-24  25-34  35-44  45-49  50-55  56+
gender                                                     
F                 0     17     18     18     17     15    4
M                15     18     18     18     18     18   17


### Step 7b: Filter the Cells Used for Ranking
Data manipulation operation: filtering. Only reliable cells are kept in the table used to rank genres within a segment; indicative cells are set aside and reported separately.

In [16]:
print("STEP 7b  filter the cells used for ranking")

ranked = cells[cells["tier"] == "reliable"]
indicative = cells[cells["tier"] == "indicative"]
assert len(ranked) + len(indicative) == len(cells), "filter lost cells"

print(f"cells before filter            : {len(cells)}")
print(f"cells kept for ranking         : {len(ranked)}")
print(f"cells set aside as indicative  : {len(indicative)}")

STEP 7b  filter the cells used for ranking
cells before filter            : 252
cells kept for ranking         : 211
cells set aside as indicative  : 41


### Step 8: Satisfied Flag and Satisfaction Rate per Cell
Derived column `satisfied` (rating ≥ 4) and the satisfaction rate of each cell, as defined in D1 Section 2: the share of a cell's **ratings** that are 4 or 5 stars. No confidence interval is reported here: ratings from one user are not independent, so an interval that treats each rating as independent would be too narrow. User-level intervals are part of the D3 analysis. Rows below are sorted by segment and genre, not by rate; no genre is ranked in D2.

In [17]:
print("STEP 8  satisfied flag and satisfaction rate per cell")

exploded["satisfied"] = exploded["rating"] >= 4

rates = (
    exploded.groupby(["gender", "age_group", "genre"], observed=True)
    .agg(
        n_ratings=("rating", "size"),
        n_users=("user_id", "nunique"),
        n_satisfied=("satisfied", "sum"),
        sat_rate=("satisfied", "mean"),
    )
    .reset_index()
)

rates = rates.merge(
    cells[["gender", "age_group", "genre", "tier"]],
    on=["gender", "age_group", "genre"],
    how="left",
    validate="1:1",
)

assert len(rates) == len(cells), "cell count changed"
assert rates["sat_rate"].between(0, 1).all(), "sat_rate outside [0, 1]"
assert (rates["n_satisfied"] <= rates["n_ratings"]).all(), "more satisfied than ratings"

print(f"cells                                     : {len(rates)}")
print(f"share of ratings >= 4, all ratings, no explode : {(df['rating'] >= 4).mean():.4f}")
print("\nfirst rows of the cell table (sorted by segment and genre, not by rate)")
print(rates.sort_values(["gender", "age_group", "genre"]).head(5).to_string(index=False))

STEP 8  satisfied flag and satisfaction rate per cell


cells                                     : 252
share of ratings >= 4, all ratings, no explode : 0.5752

first rows of the cell table (sorted by segment and genre, not by rate)
gender age_group      genre  n_ratings  n_users  n_satisfied  sat_rate     tier
     F     18-24     Action       8593      296         4126  0.480158 reliable
     F     18-24  Adventure       5430      291         2611  0.480847 reliable
     F     18-24  Animation       3109      259         1806  0.580894 reliable
     F     18-24 Children's       5234      276         2575  0.491976 reliable
     F     18-24     Comedy      19136      298         9878  0.516200 reliable


### Step 9: Data Profile of the Working Table
Rows, columns, types, missing values, distinct values and ranges for the merged table `df` (after dropping personal columns and adding `year` and `age_group`).

In [18]:
print("STEP 9  data profile of the working table")

_ = profile(df, "df (working table)")

STEP 9  data profile of the working table



df (working table): 1,000,209 rows x 10 columns
   column dtype  missing  unique       min        max
  user_id int64        0    6040         1       6040
 movie_id int64        0    3706         1       3952
   rating int64        0       5         1          5
timestamp int64        0  458455 956703932 1046454590
   gender   str        0       2         -          -
      age int64        0       7         1         56
    title   str        0    3706         -          -
   genres   str        0     301         -          -
     year Int64        0      81      1919       2000
age_group   str        0       7         -          -


## D3 analysis

### Step B1: Split Users into a Discovery Half and a Validation Half
Within each of the 14 segments, users are shuffled with seed 42 and split 50/50. The split is made on users, not on ratings, so all ratings of one person stay in the same half (ratings from one user are not independent). When a segment has an odd number of users, the extra user goes to the validation half. Genres are chosen on the discovery half (Steps B4 to B7) and tested on the validation half (Step B8), so the noise that helped a genre get selected is not counted again in the test. Only counts are printed here; no satisfaction rate is computed per half in this Step. Tiers stay as frozen in Step 7 on the full data.

In [19]:
print("STEP B1  split users into a discovery half and a validation half")

rng = np.random.default_rng(SEED)

seg_users = users[["user_id", "gender", "age"]].copy()
seg_users["age_group"] = seg_users["age"].map(AGE_LABEL)

half_of: dict[int, str] = {}
for gender in ["F", "M"]:              # fixed order, so the random draws are reproducible
    for age in age_order:
        in_seg = (seg_users["gender"] == gender) & (seg_users["age_group"] == age)
        ids = np.sort(seg_users.loc[in_seg, "user_id"].to_numpy())
        shuffled = rng.permutation(ids)
        n_discovery = len(ids) // 2    # odd count: the extra user goes to validation
        for uid in shuffled[:n_discovery]:
            half_of[int(uid)] = "discovery"
        for uid in shuffled[n_discovery:]:
            half_of[int(uid)] = "validation"

seg_users["half"] = seg_users["user_id"].map(half_of)
df["half"] = df["user_id"].map(half_of)
exploded["half"] = exploded["user_id"].map(half_of)

assert len(half_of) == len(users) == seg_users["user_id"].nunique(), "a user is missing or repeated"
assert seg_users["half"].notna().all(), "a user has no half"
assert df["half"].notna().all() and exploded["half"].notna().all(), "a rating has no half"
assert (df.groupby("user_id")["half"].nunique() == 1).all(), "a user appears in both halves"

users_per_half = (
    seg_users.groupby(["gender", "age_group", "half"]).size()
    .unstack("half")
    .reindex(pd.MultiIndex.from_product([["F", "M"], age_order], names=["gender", "age_group"]))
)
users_per_half["difference"] = users_per_half["validation"] - users_per_half["discovery"]
assert users_per_half["difference"].isin([0, 1]).all(), "halves differ by more than one user"

print("users per segment and half (difference = validation - discovery)")
print(users_per_half.to_string())
print(f"\nusers, discovery  : {(seg_users['half'] == 'discovery').sum():,}")
print(f"users, validation : {(seg_users['half'] == 'validation').sum():,}")
print(f"users, total      : {len(seg_users):,}")

ratings_per_half = df["half"].value_counts().reindex(["discovery", "validation"])
print("\nratings per half (working table df)")
for half, n in ratings_per_half.items():
    print(f"  {half:<10} : {n:,}")
print(f"  total      : {ratings_per_half.sum():,}")

STEP B1  split users into a discovery half and a validation half


users per segment and half (difference = validation - discovery)
half              discovery  validation  difference
gender age_group                                   
F      Under 18          39          39           0
       18-24            149         149           0
       25-34            279         279           0
       35-44            169         169           0
       45-49             94          95           1
       50-55             73          73           0
       56+               51          51           0
M      Under 18          72          72           0
       18-24            402         403           1
       25-34            769         769           0
       35-44            427         428           1
       45-49            180         181           1
       50-55            175         175           0
       56+              139         139           0

users, discovery  : 3,018
users, validation : 3,022
users, total      : 6,040

ratings per half (worki

## Data analyst A: independent replication (Steps A2 to A7) and tables (Steps Q1, Q2)
Written without looking at the code of Data analyst B. Steps A2 to A7 follow analysis steps 2 to 7 of the D3 procedure and are used only to cross-check B, T(s) and the segment categories with Data analyst B; they are not merged into the final notebook. Steps Q1 and Q2 are merged and use the column names of `D3_columns_HoangQuan_v2`. Everything below starts from the objects of Steps 1 to B1 (`users`, `exploded`, `seg_users`, `cells`, `age_order`, `AGE_LABEL`, `SEED`).

### Step A2: Count n(u, g) and k(u, g) for every user and genre
For each user u and genre g, `N[u, g]` is the number of ratings that u gave to films of genre g, and `K[u, g]` is how many of them are 4 or 5 stars. A user who never rated genre g has `N = K = 0`. Both matrices have one row per user (6,040) and one column per genre (18). Every later Step works only on these two matrices, so one user's ratings always travel together.

In [20]:
print("STEP A2  n(u, g) and k(u, g) per user and genre")

GENRES = sorted(exploded["genre"].unique())
SEGMENTS = [(g, a) for g in ["F", "M"] for a in age_order]      # fixed order: F then M, ages young to old
user_ids = np.sort(users["user_id"].to_numpy())
row_of_user = pd.Series(np.arange(len(user_ids)), index=user_ids)
col_of_genre = {g: j for j, g in enumerate(GENRES)}

rows = exploded["user_id"].map(row_of_user).to_numpy()
cols = exploded["genre"].map(col_of_genre).to_numpy()
flat = rows * len(GENRES) + cols
size = len(user_ids) * len(GENRES)
N = np.bincount(flat, minlength=size).reshape(len(user_ids), len(GENRES))
K = np.bincount(flat, weights=(exploded["rating"] >= 4).to_numpy().astype(float), minlength=size)
K = K.round().astype(np.int64).reshape(len(user_ids), len(GENRES))

assert N.sum() == len(exploded) == 2_101_815, "N does not add up to the exploded table"
assert (K <= N).all(), "more satisfied ratings than ratings"
print(f"matrix shape        : {N.shape} (users x genres)")
print(f"sum of N            : {N.sum():,} (rows of the exploded table)")
print(f"sum of K            : {K.sum():,}")
print(f"share of zero N     : {(N == 0).mean():.1%} of the user x genre entries")

# user positions of every segment and half (the half comes from Step B1)
seg_users["row"] = seg_users["user_id"].map(row_of_user)
members = {
    (g, a, h): seg_users.loc[
        (seg_users["gender"] == g) & (seg_users["age_group"] == a) & (seg_users["half"] == h), "row"
    ].to_numpy()
    for (g, a) in SEGMENTS for h in ("discovery", "validation")
}
assert sum(len(v) for v in members.values()) == len(users)

STEP A2  n(u, g) and k(u, g) per user and genre


matrix shape        : (6040, 18) (users x genres)
sum of N            : 2,101,815 (rows of the exploded table)
sum of K            : 1,206,999
share of zero N     : 13.6% of the user x genre entries


### Step A3: Satisfaction rate and counts of every cell, in each half
For a segment s and a half, `r(s, g) = sum of K[u, g] / sum of N[u, g]` over the users u of the segment in that half. Besides the rate, the Step records the number of ratings, distinct users, satisfied ratings and the reach (distinct users of the cell divided by users of the segment in the same half). The tier is the one frozen in Step 7 on all data; the count of distinct users on all data is recomputed here from `N` as a cross-check of Step 7.

In [21]:
print("STEP A3  satisfaction rate and counts per cell and half")

records = []
for (g, a) in SEGMENTS:
    all_rows = np.concatenate([members[(g, a, "discovery")], members[(g, a, "validation")]])
    n_users_all = (N[all_rows] > 0).sum(axis=0)
    for half in ("discovery", "validation"):
        r = members[(g, a, half)]
        n_rat = N[r].sum(axis=0)
        n_sat = K[r].sum(axis=0)
        n_usr = (N[r] > 0).sum(axis=0)
        for j, genre in enumerate(GENRES):
            records.append({
                "gender": g, "age_group": a, "genre": genre, "half": half,
                "n_ratings": int(n_rat[j]), "n_users": int(n_usr[j]), "n_satisfied": int(n_sat[j]),
                "sat_rate": n_sat[j] / n_rat[j] if n_rat[j] > 0 else np.nan,
                "n_users_all": int(n_users_all[j]),
                "segment_users": len(r),
            })

cell_half = pd.DataFrame(records)
cell_half["tier"] = np.where(cell_half["n_users_all"] >= MIN_USERS, "reliable", "indicative")
cell_half["reach"] = cell_half["n_users"] / cell_half["segment_users"]

# cross-check with Steps 6 and 7 (computed on all data with another method)
check = cell_half[cell_half["half"] == "discovery"].merge(
    cells[["gender", "age_group", "genre", "n_users", "tier"]].rename(columns={"n_users": "n_users_step6", "tier": "tier_step7"}),
    on=["gender", "age_group", "genre"], validate="1:1")
assert len(check) == 252
assert (check["n_users_all"] == check["n_users_step6"]).all(), "distinct users differ from Step 6"
assert (check["tier"] == check["tier_step7"]).all(), "tier differs from Step 7"
assert cell_half["n_ratings"].sum() == len(exploded), "ratings of the two halves do not add up to the exploded table"

print(f"rows in cell_half : {len(cell_half)} (14 segments x 18 genres x 2 halves)")
print(f"ratings, discovery  : {cell_half.loc[cell_half['half']=='discovery', 'n_ratings'].sum():,}")
print(f"ratings, validation : {cell_half.loc[cell_half['half']=='validation', 'n_ratings'].sum():,}")
print("n_users_all and tier agree with Steps 6 and 7 in all 252 cells")
print(cell_half[cell_half["half"] == "discovery"]["tier"].value_counts().to_string())

STEP A3  satisfaction rate and counts per cell and half
rows in cell_half : 504 (14 segments x 18 genres x 2 halves)
ratings, discovery  : 1,061,414
ratings, validation : 1,040,401
n_users_all and tier agree with Steps 6 and 7 in all 252 cells
tier
reliable      211
indicative     41


### Step A4: LCB of every cell on the discovery half
Bootstrap over users: within one segment, draw as many users as the discovery half has, with replacement, bring all their ratings, recompute the rate of every genre. Repeat 1,000 times. `LCB(s, g)` is the 2.5% quantile (`np.quantile`, linear interpolation) of the 1,000 rates. The random generator is created at the start of the Step with seed 42; segments are visited in the fixed order F then M, ages young to old, and the same drawn users are used for all 18 genres of a segment.

In [22]:
print("STEP A4  bootstrap LCB of every cell, discovery half")

N_BOOT = 1000
rng = np.random.default_rng(SEED)

def boot_rates(Nm, Km, n_boot, rng):
    """n_boot x genres matrix of rates; users resampled with replacement, all ratings of a user kept together."""
    n = Nm.shape[0]
    out = np.full((n_boot, Nm.shape[1]), np.nan)
    for b in range(n_boot):
        mult = np.bincount(rng.integers(0, n, size=n), minlength=n)   # how many times each user was drawn
        den = mult @ Nm
        num = mult @ Km
        ok = den > 0
        out[b, ok] = num[ok] / den[ok]
    return out

lcb_rows = []
for (g, a) in SEGMENTS:
    r = members[(g, a, "discovery")]
    draws = boot_rates(N[r], K[r], N_BOOT, rng)
    lcb = np.nanquantile(draws, 0.025, axis=0)
    for j, genre in enumerate(GENRES):
        lcb_rows.append({"gender": g, "age_group": a, "genre": genre, "lcb": lcb[j],
                         "n_valid_draws": int(np.isfinite(draws[:, j]).sum())})

lcb_table = pd.DataFrame(lcb_rows)
disc = cell_half[cell_half["half"] == "discovery"].merge(lcb_table, on=["gender", "age_group", "genre"], validate="1:1")
assert (disc["lcb"] <= disc["sat_rate"] + 1e-9).all(), "an LCB is above its own rate"
print(f"cells with an LCB : {disc['lcb'].notna().sum()} of {len(disc)}")
print(f"cells where some draws had no rating for the genre : {(disc['n_valid_draws'] < N_BOOT).sum()}")
print("gap sat_rate - lcb, reliable cells vs indicative cells (mean, in percentage points)")
print(((disc['sat_rate'] - disc['lcb']) * 100).groupby(disc['tier']).mean().round(2).to_string())

STEP A4  bootstrap LCB of every cell, discovery half


cells with an LCB : 252 of 252
cells where some draws had no rating for the genre : 1
gap sat_rate - lcb, reliable cells vs indicative cells (mean, in percentage points)
tier
indicative    11.93
reliable       5.30


### Step A5: T(s), the three reliable cells with the highest LCB in each segment
Only reliable cells are ranked. Ties: more distinct users first (`n_users_all`), then genre name in alphabetical order. A segment without reliable cells (female under 18) has no T(s).

In [23]:
print("STEP A5  choose T(s)")

disc["rank_lcb"] = pd.array([pd.NA] * len(disc), dtype="Int64")
T = {}
for (g, a) in SEGMENTS:
    in_seg = (disc["gender"] == g) & (disc["age_group"] == a) & (disc["tier"] == "reliable")
    ranked_seg = disc[in_seg].sort_values(["lcb", "n_users_all", "genre"], ascending=[False, False, True])
    disc.loc[ranked_seg.index, "rank_lcb"] = np.arange(1, len(ranked_seg) + 1)
    T[(g, a)] = ranked_seg["genre"].head(3).tolist() if len(ranked_seg) >= 3 else []
disc["in_top3"] = disc["rank_lcb"].notna() & (disc["rank_lcb"].astype("float") <= 3)

# a tie in LCB would be settled by the tie rule; count how often the rule was needed among the ranked cells
ties = (disc[disc["tier"] == "reliable"].groupby(["gender", "age_group"])["lcb"]
        .apply(lambda s: int(s.duplicated().sum())).sum())
print(f"tied LCB values among reliable cells : {ties}")
for (g, a), genres_ in T.items():
    print(f"  {g} {a:<9}: {'; '.join(genres_) if genres_ else '(no T(s): no reliable cell)'}")

STEP A5  choose T(s)
tied LCB values among reliable cells : 0
  F Under 18 : (no T(s): no reliable cell)
  F 18-24    : Film-Noir; War; Musical
  F 25-34    : Film-Noir; Documentary; War
  F 35-44    : Film-Noir; War; Musical
  F 45-49    : War; Musical; Drama
  F 50-55    : War; Drama; Romance
  F 56+      : Drama; Romance; Thriller
  M Under 18 : War; Drama; Crime
  M 18-24    : Film-Noir; War; Documentary
  M 25-34    : Film-Noir; Documentary; War
  M 35-44    : Film-Noir; Documentary; War
  M 45-49    : Film-Noir; War; Crime
  M 50-55    : Film-Noir; War; Mystery
  M 56+      : Film-Noir; War; Musical


### Step A6: Baseline B
The 12 reference segments are the segments with at least 15 reliable cells (read from the tier table, not typed by hand). The candidate genres are the genres that are reliable in all 12 of them. Weights: `w(s) = N(s) / sum of N(s)` over the 12 segments, where `N(s)` is the number of users of the segment in the full `users` table. Score: `S(g) = sum over s of w(s) * r(s, g)` with `r` the discovery-half rate. Bootstrap 1,000 times: users are resampled inside each of the 12 segments, `w(s)` stays fixed; `LCB(S(g))` is the 2.5% quantile of the 1,000 values of `S(g)`. B is the three candidate genres with the highest `LCB(S(g))`; ties: more distinct users first (discovery half, summed over the 12 segments), then alphabetical.

In [24]:
print("STEP A6  baseline B")

rel = cells.groupby(["gender", "age_group"], observed=False)["tier"].apply(lambda s: int((s == "reliable").sum()))
REF = [(g, a) for (g, a) in SEGMENTS if rel[(g, a)] >= 15]
print(f"reference segments : {len(REF)}")
print("excluded           : " + ", ".join(f"{g} {a} ({rel[(g, a)]} reliable cells)" for (g, a) in SEGMENTS if (g, a) not in REF))

tier_of = disc.set_index(["gender", "age_group", "genre"])["tier"]
CANDIDATES = [x for x in GENRES if all(tier_of[(g, a, x)] == "reliable" for (g, a) in REF)]
cand_idx = [col_of_genre[g] for g in CANDIDATES]
print(f"candidate genres   : {len(CANDIDATES)}  (excluded: {[g for g in GENRES if g not in CANDIDATES]})")

N_seg = {s: int(((users['gender'] == s[0]) & (users['age'].map(AGE_LABEL) == s[1])).sum()) for s in REF}
w = {s: N_seg[s] / sum(N_seg.values()) for s in REF}
assert abs(sum(w.values()) - 1) < 1e-12

# score on the data
S = np.zeros(len(cand_idx))
for s in REF:
    r = members[(s[0], s[1], "discovery")]
    S += w[s] * (K[r].sum(axis=0)[cand_idx] / N[r].sum(axis=0)[cand_idx])

# bootstrap of S(g): users resampled inside each segment, weights fixed
rng = np.random.default_rng(SEED)
S_boot = np.zeros((N_BOOT, len(cand_idx)))
for s in REF:
    r = members[(s[0], s[1], "discovery")]
    S_boot += w[s] * boot_rates(N[r][:, cand_idx], K[r][:, cand_idx], N_BOOT, rng)
S_lcb = np.quantile(S_boot, 0.025, axis=0)

users_disc = np.array([sum(int((N[members[(s[0], s[1], 'discovery')]][:, j] > 0).sum()) for s in REF) for j in cand_idx])
baseline = pd.DataFrame({"genre": CANDIDATES, "s_g": S, "lcb": S_lcb, "n_users_ref": users_disc})
baseline = baseline.sort_values(["lcb", "n_users_ref", "genre"], ascending=[False, False, True]).reset_index(drop=True)
baseline["rank_lcb"] = np.arange(1, len(baseline) + 1)
baseline["in_baseline"] = baseline["rank_lcb"] <= 3
B_SET = baseline.loc[baseline["in_baseline"], "genre"].tolist()

print("\nbaseline table (15 candidate genres)")
print(baseline.drop(columns="n_users_ref").round(4).to_string(index=False))
print(f"\nB = {'; '.join(B_SET)}")

STEP A6  baseline B
reference segments : 12
excluded           : F Under 18 (0 reliable cells), F 56+ (4 reliable cells)
candidate genres   : 15  (excluded: ['Documentary', 'Film-Noir', 'Western'])



baseline table (15 candidate genres)
     genre    s_g    lcb  rank_lcb  in_baseline
       War 0.6907 0.6810         1         True
     Drama 0.6404 0.6298         2         True
     Crime 0.6194 0.6082         3         True
 Animation 0.6192 0.6063         4        False
   Musical 0.6009 0.5865         5        False
   Mystery 0.5938 0.5798         6        False
   Romance 0.5748 0.5636         7        False
  Thriller 0.5644 0.5538         8        False
    Comedy 0.5535 0.5432         9        False
    Action 0.5368 0.5268        10        False
 Adventure 0.5313 0.5210        11        False
    Sci-Fi 0.5241 0.5143        12        False
Children's 0.5221 0.5087        13        False
   Fantasy 0.5153 0.5035        14        False
    Horror 0.4386 0.4253        15        False

B = War; Drama; Crime


### Step A7: Classify every segment
Decided from the tier and the discovery half only, before any test on the validation half. In this order: (1) *insufficient evidence* if the segment has no T(s) (no reliable cell), or if at least one genre of B is indicative in the segment; (2) *same as baseline* if T(s) is exactly the set B; (3) otherwise the segment is *tested* (later *passed* or *not passed*). `m` is the number of segments tested.

In [25]:
print("STEP A7  classify the segments")

tier_of = disc.set_index(["gender", "age_group", "genre"])["tier"]
class_rows = []
for (g, a) in SEGMENTS:
    ts = T[(g, a)]
    ind_in_b = [x for x in B_SET if tier_of[(g, a, x)] == "indicative"]
    if not ts:
        cat, reason = "insufficient_evidence", "no reliable cell in the segment"
    elif ind_in_b:
        cat, reason = "insufficient_evidence", "genre of B indicative in the segment: " + "; ".join(ind_in_b)
    elif set(ts) == set(B_SET):
        cat, reason = "same_as_baseline", "T(s) equals B"
    else:
        cat, reason = "to_test", ""
    class_rows.append({"segment": f"{g} {a}", "gender": g, "age_group": a, "category": cat, "reason": reason,
                       "t_s": "; ".join(ts)})

classification = pd.DataFrame(class_rows)
M_TESTED = int((classification["category"] == "to_test").sum())
classification["m"] = M_TESTED
print(classification[["segment", "category", "t_s", "reason"]].to_string(index=False))
print(f"\nsegments by category:\n{classification['category'].value_counts().to_string()}")
print(f"\nm (segments to be tested) = {M_TESTED}")

STEP A7  classify the segments
   segment              category                         t_s                                           reason
F Under 18 insufficient_evidence                                              no reliable cell in the segment
   F 18-24               to_test     Film-Noir; War; Musical                                                 
   F 25-34               to_test Film-Noir; Documentary; War                                                 
   F 35-44               to_test     Film-Noir; War; Musical                                                 
   F 45-49               to_test         War; Musical; Drama                                                 
   F 50-55               to_test         War; Drama; Romance                                                 
     F 56+ insufficient_evidence    Drama; Romance; Thriller genre of B indicative in the segment: War; Crime
M Under 18      same_as_baseline           War; Drama; Crime                             

### Step Q1: Descriptive statistics of the 252 cells on the discovery half
One row per cell (segment × genre) with the number of ratings, distinct users, satisfied ratings, satisfaction rate, reach and tier. Column names follow Table 1 of `D3_columns_HoangQuan_v2`; `half` is `discovery` in every row. The table describes the data used to choose genres; it does not rank or test anything.

In [26]:
print("STEP Q1  descriptive statistics, 252 cells, discovery half")

q1_table = (
    cell_half[cell_half["half"] == "discovery"]
    [["gender", "age_group", "genre", "half", "n_ratings", "n_users", "n_satisfied", "sat_rate",
      "n_users_all", "tier", "segment_users", "reach"]]
    .assign(age_group=lambda d: pd.Categorical(d["age_group"], age_order, ordered=True))
    .sort_values(["gender", "age_group", "genre"])
    .reset_index(drop=True)
)
q1_table["age_group"] = q1_table["age_group"].astype(str)
assert len(q1_table) == 252 and q1_table["n_ratings"].sum() == cell_half.loc[cell_half["half"] == "discovery", "n_ratings"].sum()

print(f"cells: {len(q1_table)}; ratings in the cells: {q1_table['n_ratings'].sum():,} (a rating of a multi-genre film is counted once per genre)")
desc = q1_table[["n_ratings", "n_users", "n_satisfied", "sat_rate", "reach"]].describe().T
print("\nsummary of the 252 cells")
print(desc.round(3).to_string())

print("\nsummary by tier (cells, median users, median rate, median reach)")
print(q1_table.groupby("tier").agg(cells=("genre", "size"), median_users=("n_users", "median"),
                                    median_rate=("sat_rate", "median"), median_reach=("reach", "median")).round(3).to_string())

print("\nsatisfaction rate by genre, discovery half, all cells (min / median / max)")
print(q1_table.groupby("genre")["sat_rate"].agg(["min", "median", "max"]).round(3).sort_values("median", ascending=False).to_string())

q1_table.round(4)

STEP Q1  descriptive statistics, 252 cells, discovery half
cells: 252; ratings in the cells: 1,061,414 (a rating of a multi-genre film is counted once per genre)

summary of the 252 cells
             count      mean       std     min      25%       50%       75%        max
n_ratings    252.0  4211.960  7328.687  13.000  625.750  1575.500  4435.750  51787.000
n_users      252.0   186.611   177.395   6.000   66.750   136.500   227.500    769.000
n_satisfied  252.0  2406.925  4119.032   9.000  411.250   961.500  2588.250  31806.000
sat_rate     252.0     0.609     0.091   0.395    0.541     0.601     0.673      0.877
reach        252.0     0.849     0.179   0.154    0.770     0.911     0.986      1.000

summary by tier (cells, median users, median rate, median reach)
            cells  median_users  median_rate  median_reach
tier                                                      
indicative     41          38.0        0.678         0.795
reliable      211         149.0        0.590   

,gender,age_group,genre,half,n_ratings,n_users,n_satisfied,sat_rate,n_users_all,tier,segment_users,reach
0,F,Under 18,Action,discovery,988,39,572,0.5789,77,indicative,39,1.0000
1,F,Under 18,Adventure,discovery,833,37,463,0.5558,73,indicative,39,0.9487
2,F,Under 18,Animation,discovery,568,35,362,0.6373,68,indicative,39,0.8974
3,F,Under 18,Children's,discovery,1119,38,609,0.5442,73,indicative,39,0.9744
4,F,Under 18,Comedy,discovery,2171,39,1201,0.5532,78,indicative,39,1.0000
...,...,...,...,...,...,...,...,...,...,...,...,...
247,M,56+,Romance,discovery,2258,135,1467,0.6497,271,reliable,139,0.9712
248,M,56+,Sci-Fi,discovery,1949,129,956,0.4905,266,reliable,139,0.9281
249,M,56+,Thriller,discovery,2599,138,1483,0.5706,273,reliable,139,0.9928
250,M,56+,War,discovery,1453,131,1056,0.7268,262,reliable,139,0.9424


### Step Q2: Summary table, T(s) with rate, LCB and reach, plus the baseline B and the category of every segment
For each segment: the three genres of T(s) with their discovery-half satisfaction rate, LCB, reach and rank, and the category from Step A7. The last three rows are the baseline B: they use the columns of Table 3 (`s_g` is the weighted score, `lcb` its LCB, `rank_lcb` its rank, `in_baseline`); `reach` and `sat_rate` are not defined for B and left empty. Segments without T(s) appear with one row and empty genre.

In [27]:
print("STEP Q2  summary table")

disc_idx = disc.set_index(["gender", "age_group", "genre"])
cat_of = classification.set_index("segment")
q2_rows = []
for (g, a) in SEGMENTS:
    seg = f"{g} {a}"
    ts = T[(g, a)]
    base = {"segment": seg, "gender": g, "age_group": a, "category": cat_of.loc[seg, "category"], "role": "T(s)"}
    if not ts:
        q2_rows.append({**base, "genre": None})
    for x in ts:
        row = disc_idx.loc[(g, a, x)]
        q2_rows.append({**base, "genre": x, "tier": row["tier"], "n_users": int(row["n_users"]),
                        "reach": row["reach"], "sat_rate": row["sat_rate"], "lcb": row["lcb"],
                        "rank_lcb": int(row["rank_lcb"]), "in_top3": True})
for _, b in baseline[baseline["in_baseline"]].iterrows():
    q2_rows.append({"segment": "all (baseline B)", "category": "baseline", "role": "B", "genre": b["genre"],
                    "tier": "reliable", "s_g": b["s_g"], "lcb": b["lcb"], "rank_lcb": int(b["rank_lcb"]),
                    "in_baseline": True})

q2_cols = ["segment", "gender", "age_group", "category", "role", "genre", "tier", "n_users", "reach",
           "sat_rate", "lcb", "rank_lcb", "in_top3", "s_g", "in_baseline"]
q2_table = pd.DataFrame(q2_rows).reindex(columns=q2_cols)
q2_table["rank_lcb"] = q2_table["rank_lcb"].astype("Int64")
q2_table["n_users"] = q2_table["n_users"].astype("Int64")
print(q2_table.drop(columns=["gender", "age_group"]).round(4).to_string(index=False, na_rep=""))

STEP Q2  summary table
         segment              category role       genre     tier  n_users  reach  sat_rate    lcb  rank_lcb in_top3    s_g in_baseline
      F Under 18 insufficient_evidence T(s)                          <NA>                              <NA>                           
         F 18-24               to_test T(s)   Film-Noir reliable       86 0.5772    0.7050 0.6250         1    True                   
         F 18-24               to_test T(s)         War reliable      142 0.9530    0.6641 0.6200         2    True                   
         F 18-24               to_test T(s)     Musical reliable      124 0.8322    0.6530 0.6005         3    True                   
         F 25-34               to_test T(s)   Film-Noir reliable      169 0.6057    0.7279 0.6820         1    True                   
         F 25-34               to_test T(s) Documentary reliable      102 0.3656    0.7462 0.6802         2    True                   
         F 25-34               t

### Step A8: Export B, T(s) and the categories for the cross-check with Data analyst B
One row per segment (14 rows): segment, gender, age_group, category, reason, t_s (genres separated by `; `), b (the baseline, same in every row) and m. Only B, T(s) and the category are compared with Data analyst B; LCB values are not compared digit by digit because two independent bootstrap codes differ slightly.

In [28]:
print("STEP A8  export the check file")

check_file = classification.copy()
check_file["b"] = "; ".join(B_SET)
check_file = check_file[["segment", "gender", "age_group", "category", "reason", "t_s", "b", "m"]]
check_file.to_csv("D3_check_Que.csv", index=False, encoding="utf-8")

print(pd.read_csv("D3_check_Que.csv").to_string(index=False))
print(f"\nwritten: D3_check_Que.csv, {len(check_file)} rows")

STEP A8  export the check file
   segment gender age_group              category                                           reason                         t_s                 b  m
F Under 18      F  Under 18 insufficient_evidence                  no reliable cell in the segment                         NaN War; Drama; Crime 11
   F 18-24      F     18-24               to_test                                              NaN     Film-Noir; War; Musical War; Drama; Crime 11
   F 25-34      F     25-34               to_test                                              NaN Film-Noir; Documentary; War War; Drama; Crime 11
   F 35-44      F     35-44               to_test                                              NaN     Film-Noir; War; Musical War; Drama; Crime 11
   F 45-49      F     45-49               to_test                                              NaN         War; Musical; Drama War; Drama; Crime 11
   F 50-55      F     50-55               to_test                                

## Data manipulation operations (index)
| Operation | Where | What it does |
|-----------|-------|--------------|
| Merging two tables | Step 3 | left joins ratings, users and movies with validate="m:1" and asserts the row count |
| Creating a derived column | Steps 4, 7, 8 | year and age_group (Step 4), tier (Step 7), satisfied flag (Step 8) |
| Filtering | Step 7b | keeps the reliable cells (≥ 100 distinct users) used for ranking, counts printed before and after |
| Grouping and aggregation | Steps 6, 8 | users, ratings and satisfaction rate per gender x age_group x genre cell |

Step 5 (explode genres into one row per rating-genre pair) prepares the data for Steps 6 and 8 and is not counted as a fifth operation.

### Things that do NOT change the row count
- Steps 5 and 6 never modify `df`; they build `exploded` and the cell table from it. Cell sizes are counted in distinct users, and a user who rated several genres is counted in several cells of the same segment, so the cell sizes inside one segment do not add up to that segment's number of users.

## Inputs for the other roles
Environment and column lists printed from this notebook, so teammates take them from here instead of typing them by hand.

In [29]:
print("HANDOFF  environment and column lists")

import platform
from datetime import date
from importlib.metadata import version

print(f"run date: {date.today().isoformat()}")
print(f"python  : {platform.python_version()}")
for pkg in ("numpy", "pandas"):
    print(f"{pkg:<8}: {version(pkg)}")
print(f"seed    : {SEED}")

print("\nworking table df, columns")
for col, dtype in df.dtypes.items():
    print(f"  {col:<12} {dtype}")

print("\nexploded table, columns")
for col, dtype in exploded.dtypes.items():
    print(f"  {col:<12} {dtype}")

print("\ncell table rates, columns")
for col, dtype in rates.dtypes.items():
    print(f"  {col:<12} {dtype}")

HANDOFF  environment and column lists
run date: 2026-11-18
python  : 3.13.18
numpy   : 2.5.3
pandas  : 3.0.5
seed    : 42

working table df, columns
  user_id      int64
  movie_id     int64
  rating       int64
  timestamp    int64
  gender       str
  age          int64
  title        str
  genres       str
  year         Int64
  age_group    str
  half         str

exploded table, columns
  user_id      int64
  movie_id     int64
  rating       int64
  timestamp    int64
  gender       str
  age          int64
  title        str
  genres       str
  year         Int64
  age_group    str
  genre        str
  n_genres     int64
  satisfied    bool
  half         str

cell table rates, columns
  gender       str
  age_group    object
  genre        str
  n_ratings    int64
  n_users      int64
  n_satisfied  int64
  sat_rate     float64
  tier         str


## Cleaning log

Every figure below is printed by the Step named in the "Step" column. Rows C1 to C17 come from the D2 run of 2026-10-20 and every rerun must print the same figures; C18 and C19 were added in D3. Row counts refer to the 1,000,209-row ratings table unless stated otherwise.

| ID | Step | Check | Finding | Decision | Reason | Before → After |
|----|------|-------|---------|----------|--------|----------------|
| C1 | 2 | Missing values in `ratings`, `users`, `movies` | 0 missing in every column of all three tables | Keep as is | Nothing to impute or drop | 1,000,209 / 6,040 / 3,883 rows → unchanged |
| C2 | 2 | Primary key uniqueness | `users.user_id` and `movies.movie_id` are unique | Keep as is | Joins are many-to-one, so no row can multiply | unchanged |
| C3 | 2 | Orphan foreign keys in `ratings` | 0 rows with an unknown `user_id`, 0 with an unknown `movie_id` | Keep as is | Every rating can be joined | unchanged |
| C4 | 2c | Duplicate ratings | 0 repeated (`user_id`, `movie_id`) pairs, 0 fully identical rows | Nothing removed | No duplicates exist | 1,000,209 → 1,000,209 |
| C5 | 2c | Valid value ranges | `rating` ∈ {1,2,3,4,5}; `gender` ∈ {F, M}; ratings dated 2000-04-25 to 2003-02-28 | Keep as is | All values are inside the documented domain | unchanged |
| C6 | 2c | Films nobody rated | 3,883 films in `movies.dat`, 3,706 with at least 1 rating, 177 never rated | Left joins start from `ratings`, so the 177 films do not appear in the working table | A film with no rating carries no preference signal | 3,883 → 3,706 films in `df` |
| C7 | 3 | Merge `ratings` + `users` + `movies` | Left joins with `validate="m:1"`; row count asserted equal to `ratings` | Merged into one working table | One row per rating with user and film attributes | 1,000,209 → 1,000,209 rows; 4 → 10 columns |
| C8 | 3b | Personal columns | `zip` (3,439 distinct) could single out a person when combined with gender and age band; `occupation` is not used in the analysis | Dropped `zip` and `occupation` | Privacy and relevance, as committed in D1 | 10 → 8 columns |
| C9 | 4 / 4b | Release year parsed from `title` | 0 unparsed titles; years 1919–2000; 81 distinct years in rated films and in `movies.dat`; the only year without a film inside the span is 1924 | Added `year` | Needed for any release-period cut | 8 → 9 columns |
| C10 | 4 | Age code to label | 0 unmapped codes; 7 groups: Under 18 (27,211), 18-24 (183,536), 25-34 (395,556), 35-44 (199,003), 45-49 (83,633), 50-55 (72,490), 56+ (38,780); counts sum to 1,000,209 | Added `age_group` | Readable and ordered labels for the age groups | 9 → 10 columns |
| C11 | 5 | Multi-genre films | A rating row carries 1–6 genre labels (mean 2.1014); 301 distinct genre strings; 18 distinct genres | Exploded to one row per rating-genre pair, in a separate table `exploded`; `df` is untouched | The unit of analysis is the genre | 1,000,209 → 2,101,815 rows (×2.1014) |
| C12 | 6 / 7 | Sparsity of the 252 cells (2 genders × 7 age groups × 18 genres) | Distinct users per cell: min 11, median 270.5, max 1,538; 41 cells under 100 users | Tier frozen before any satisfaction rate was computed: reliable ≥ 100 distinct users, indicative < 100 | Threshold fixed in D1 Section 5, so it cannot be tuned to the outcome | 252 cells → 211 reliable / 41 indicative |
| C13 | 7b | Cells used for ranking | 211 reliable cells; the 41 indicative cells are listed in Step 6 | Kept reliable cells for ranking; indicative cells set aside and reported as indicative only, not pooled | D1 Section 5 allows “pooled or reported as indicative only”; the second option needs no extra rule | 252 → 211 cells |
| C14 | 8 | Satisfaction rate per cell | satisfied = rating ≥ 4; all rates in [0, 1] | Rate reported without a confidence interval | Ratings from one user are not independent; user-level intervals are computed in D3 | 252 cells with a rate |
| C15 | 9 | Profile of the final working table `df` | 1,000,209 rows × 10 columns, 0 missing, 81 distinct years, 7 age groups | Final | Table handed to the analysts | 10 columns |
| C16 | 2c | User activity | Fewest ratings by one user: 20 | No activity filter | GroupLens already kept only users with at least 20 ratings; heavy raters are handled by counting cell size in distinct users | unchanged |
| C17 | 2d | Additional checks (Data Analyst A, Que) | Row counts match the README (1,000,209 ratings, 6,040 users); 0 ratings dated before the film's release year; 774,028 ratings (77.4%) share the same second with another rating by the same user, largest burst 30 | Keep as is | Timestamp records when a rating was entered, often in bulk, not when the film was watched, so no conclusion in this project relies on rating time | unchanged |
| C18 | B1 | Discovery and validation split | Users of each of the 14 segments split 50/50 with seed 42; discovery 3,018 users (504,116 ratings), validation 3,022 users (496,093 ratings); halves differ by at most one user per segment | Each user is assigned to exactly one half; the extra user of an odd-sized segment goes to validation | Genres are chosen on one half and tested on the other, so selection noise is not tested on the same data; the split is by user because ratings from one user are not independent | 6,040 users → 3,018 discovery / 3,022 validation |
| C19 | 1 | Data source | Colab could not verify the GroupLens certificate on 2026-10-24 (SSL CERTIFICATE_VERIFY_FAILED); in this run Step 1 printed "source : https://files.grouplens.org/datasets/movielens/ml-1m.zip (download, attempt 1)" | Download with certifi certificates, up to 3 attempts; if all fail, read `data/ml-1m.zip` (a local copy of the same official file, never committed) | Certificate verification is never turned off; the data stays the official GroupLens file | unchanged |